<a href="https://colab.research.google.com/github/Mctiandiyou/test2/blob/main/averagedata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
"""
固定9标志物面板：患者级嵌套交叉验证（Python 3.10+）
输入：22名患者，每人3次实验，每次9条传感器波形；每条2200个电流点。
输出：每种模型/搜索策略的患者预测、指标/区间、配对比较、模型和审计记录。
重要：面板来自同一发现队列，本脚本不能消除上游面板选择偏差。
默认 MODE="smoke" 仅检查程序；正式分析改为 MODE="full"，完整运行后才可报告结果。
Bayesian 为有限候选集合上的 Gaussian process + expected improvement 搜索。
"""
from __future__ import annotations
import hashlib
import importlib.metadata
import itertools
import json
import os
import random
import re
import sys
import time
from datetime import datetime
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.stats import beta, binomtest, norm
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import (
    confusion_matrix, f1_score,
    precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.model_selection import ParameterGrid, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# ========================= 只需先检查这里 =========================
DATA_FILE = Path("/content/electrochem_data.csv")
OUTPUT_ROOT = Path("/content/drive/MyDrive/snakebite_patient_cv_results")

MODE = "full"
PATIENT_METADATA = None             # 可选 Path("patient_metadata.csv")
SEED = 42
OUTER_K, INNER_K = 5, 3
MODELS = ["LDA", "SVM", "RF", "DT", "KNN", "GB", "Logistic", "EFNet"]
SEARCHES = ["grid", "random", "bayesian"]
SEARCH_BUDGET = 4                   # random/bayesian最多候选数；grid穷举，预算可能较大
EPOCHS, PATIENCE, BATCH_SIZE = 200, 25, 8
BOOTSTRAPS = 2000
RUN_NO_AUG = True                   # 沿用相应增强CNN所选参数和轮数，配对去除增强
MAKE_PLOTS = True
MAKE_TSNE = True
AUG_FACTOR, NOISE_STD, SCALE_AMP = 2, 0.008, 0.08
SCALAR_READOUT = "peak_abs"          # 单个预定读数：每通道整个扫描的最大绝对电流
# 若实验已规定固定电位/峰区间读数，应在正式运行前修改scalar_readout，
# 并更新说明；不要看外层测试结果后挑选读数。
SENSORS = ["FGG", "FGB", "CP", "FETUB", "HBD", "CPN2", "AGT", "LUM", "ACTB"]
CLASSES = ["W", "Y", "Z"]
POINTS = 2200
# 只调少量参数；三种搜索是选择策略，并非三种新的分类器。
GRIDS = {
    "LDA": {"solver": ["lsqr"], "shrinkage": [None, "auto", 0.1, 0.5, 0.9]},
    "SVM": {"C": [0.1, 1.0, 10.0], "kernel": ["linear", "rbf"], "gamma": ["scale"]},
    "RF": {"n_estimators": [100], "max_depth": [None, 3, 5], "min_samples_leaf": [1, 2]},
    "DT": {"max_depth": [2, 3, 5], "min_samples_leaf": [1, 2]},
    "KNN": {"n_neighbors": [3, 5, 7], "weights": ["uniform", "distance"]},
    "GB": {"n_estimators": [50, 100], "learning_rate": [0.05, 0.1], "max_depth": [1, 2]},
    "Logistic": {"C": [0.01, 0.1, 1.0, 10.0, 100.0]},
    "EFNet": {"learning_rate": [3e-5, 1e-4, 3e-4], "l2": [1e-4, 1e-3]},
}
LIMITATION = (
    "Fixed-panel internal exploratory evaluation. The same 22 patients contributed "
    "to panel discovery; upstream selection bias is not removed. CIs and paired tests "
    "describe cross-fitted patient predictions and do not capture all training/selection "
    "uncertainty. Outer-score rankings are exploratory, not independent confirmation "
    "of a post-hoc selected winner."
)


def json_write(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False,
                                   default=lambda x: x.item() if isinstance(x, np.generic) else str(x)),
                          encoding="utf-8")


def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def load_data(path):
    df = pd.read_csv(path)
    df = df.drop(columns=[c for c in df if str(c).startswith("Unnamed")])
    cc = [f"current_dot{i}" for i in range(1, POINTS + 1)]
    vc = [f"voltage_dot{i}" for i in range(1, POINTS + 1)]
    missing = set(["type", "sensor_type"] + cc + vc) - set(df)
    if missing:
        raise ValueError(f"缺少列：{sorted(missing)[:10]}")
    df["type"] = df["type"].astype(str).str.strip().str.upper()
    df["sensor_type"] = df["sensor_type"].astype(str).str.strip()
    parsed = df["type"].str.extract(r"^([WYZ]\d+)_(\d+)$")
    if parsed.isna().any().any():
        raise ValueError("type必须是W1_1、Y2_3等患者_重复编号。")
    if set(df.sensor_type) != set(SENSORS):
        raise ValueError("sensor_type必须是指定9个蛋白。")
    # 仅在每条波形内部插值，不使用其他患者。
    current = df[cc].apply(pd.to_numeric, errors="coerce")
    current = current.interpolate(axis=1, limit_direction="both").ffill(axis=1).bfill(axis=1)
    voltage = df[vc].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    if not np.isfinite(current.to_numpy()).all() or not np.isfinite(voltage).all():
        raise ValueError("波形存在无法补齐的缺失值或无穷值。")
    if not np.array_equal(voltage, np.broadcast_to(voltage[0], voltage.shape)):
        raise ValueError("电压轨迹并非完全相同，需要先明确对齐方法。")
    tensors, rows = [], []
    for exp, sub in df.groupby("type", sort=True):
        if len(sub) != 9 or sub.sensor_type.nunique() != 9:
            raise ValueError(f"{exp}必须包含9个传感器且各一次。")
        # 用原始行索引从已经完成插值的矩阵提取，而不是再次读取原始缺失值。
        idx = sub.assign(_row=sub.index).set_index("sensor_type").loc[SENSORS, "_row"].to_numpy()
        tensors.append(current.loc[idx].to_numpy(np.float32).T)
        patient, replicate = exp.split("_")
        rows.append({"experiment_id": exp, "patient_id": patient,
                     "replicate_id": int(replicate), "true_class": patient[0]})
    mapping = pd.DataFrame(rows)
    if mapping.duplicated(["patient_id", "replicate_id"]).any():
        raise ValueError("重复的患者/实验编号。")
    if not (mapping.groupby("patient_id").size() == 3).all():
        raise ValueError("本分析要求每名患者恰好3次重复实验。")
    counts = mapping.drop_duplicates("patient_id").true_class.value_counts().to_dict()
    if counts != {"W": 5, "Y": 8, "Z": 9}:
        raise ValueError(f"患者数应为W/Y/Z=5/8/9，实际为{counts}")
    x = np.stack(tensors)
    y = mapping.true_class.map({c: i for i, c in enumerate(CLASSES)}).to_numpy(int)
    groups = mapping.patient_id.to_numpy()
    return x, y, groups, mapping, voltage[0]


def scalar_readout(x):
    if SCALAR_READOUT != "peak_abs":
        raise ValueError("请实现并说明选定的标量读数规则。")
    return np.max(np.abs(x), axis=1).astype(float)


def patient_splits(indices, y, groups, k, seed):
    """先在一人一行的表上分层，再映射回实验行，避免分组近似分层失败。"""
    table = pd.DataFrame({"patient": groups[indices], "y": y[indices]})
    if table.groupby("patient").y.nunique().max() != 1:
        raise ValueError("同一患者存在多个标签。")
    table = table.drop_duplicates("patient").sort_values("patient").reset_index(drop=True)
    if table.y.value_counts().min() < k:
        raise ValueError(f"每类独立患者不足以进行{k}折分层。")
    cv = StratifiedKFold(k, shuffle=True, random_state=seed)
    splits = []
    for tr, va in cv.split(table.patient, table.y):
        a = indices[np.isin(groups[indices], table.patient.iloc[tr])]
        b = indices[np.isin(groups[indices], table.patient.iloc[va])]
        assert set(groups[a]).isdisjoint(groups[b])
        assert set(y[a]) == set(y[b]) == {0, 1, 2}
        splits.append((a, b))
    return splits


def aggregate(prob, y, groups):
    if not np.isfinite(prob).all() or not np.allclose(prob.sum(1), 1, atol=1e-5):
        raise ValueError("预测概率无效。")
    ids = sorted(set(groups))
    truth = np.array([y[groups == p][0] for p in ids])
    means = np.stack([prob[groups == p].mean(0) for p in ids])
    return ids, truth, means


def macro_f1(truth, prob):
    return float(f1_score(truth, prob.argmax(1), labels=[0, 1, 2],
                          average="macro", zero_division=0))


def make_classical(name, params, train_y, train_groups, seed):
    factories = {
        "LDA": lambda: LinearDiscriminantAnalysis(**params),
        "RF": lambda: RandomForestClassifier(**params, max_features="sqrt", random_state=seed, n_jobs=1),
        "DT": lambda: DecisionTreeClassifier(**params, random_state=seed),
        "KNN": lambda: KNeighborsClassifier(**params),
        "GB": lambda: GradientBoostingClassifier(**params, random_state=seed),
        "Logistic": lambda: LogisticRegression(**params, max_iter=5000, solver="lbfgs", random_state=seed),
    }
    if name == "SVM":
        # 禁用SVC隐含的记录级概率校准；显式以患者分组校准，scaler也在校准折内拟合。
        splits = patient_splits(np.arange(len(train_y)), train_y, train_groups, 2, seed)
        base = make_pipeline(StandardScaler(), SVC(**params, probability=False, random_state=seed))
        return CalibratedClassifierCV(estimator=base, method="sigmoid", cv=splits, ensemble=False)
    return make_pipeline(StandardScaler(), factories[name]())


def build_efnet(params):
    reg = tf.keras.regularizers.l2(params["l2"])
    kernels = [15, 13, 11, 9, 7, 5, 3, 1]
    def rf(ks):
        field, jump = 1, 1
        for k in ks:
            field += k * jump
            jump *= 2
        return field
    while rf(kernels) < 2000:
        kernels.append(1)
    model = tf.keras.Sequential([tf.keras.layers.Input((POINTS, len(SENSORS)))])
    for i, k in enumerate(kernels):
        filters = [64, 96, 256][i] if i < 3 else 320
        model.add(tf.keras.layers.Conv1D(filters, k, padding="same", activation="elu", kernel_regularizer=reg))
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.MaxPooling1D(2))
    model.add(tf.keras.layers.GlobalAveragePooling1D())
    model.add(tf.keras.layers.Dense(128, activation="elu", kernel_regularizer=reg))
    model.add(tf.keras.layers.Dropout(0.6))
    model.add(tf.keras.layers.Dense(3, activation="softmax", kernel_regularizer=reg))
    model.compile(optimizer=tf.keras.optimizers.Adam(params["learning_rate"]),
                  loss="sparse_categorical_crossentropy")
    return model


def augment(x, y, seed):
    rng = np.random.default_rng(seed)
    xs, ys = [x], [y]
    for _ in range(AUG_FACTOR):
        noise = rng.normal(0, NOISE_STD, x.shape).astype(np.float32)
        scale = rng.uniform(1-SCALE_AMP, 1+SCALE_AMP, (len(x), 1, 1)).astype(np.float32)
        xs.append((x + noise) * scale)
        ys.append(y)
    xx, yy = np.concatenate(xs), np.concatenate(ys)
    order = rng.permutation(len(xx))
    return xx[order], yy[order]


class PatientValidation(tf.keras.callbacks.Callback):
    """以患者平均概率的交叉熵监控早停。只接收内层验证患者。"""
    def __init__(self, x, y, groups):
        super().__init__()
        self.x, self.y, self.groups = x, y, groups

    def on_epoch_end(self, epoch, logs=None):
        p = np.asarray(self.model(self.x, training=False))
        _, truth, prob = aggregate(p, self.y, self.groups)
        logs["val_patient_loss"] = float(-np.log(np.clip(prob[np.arange(len(truth)), truth], 1e-12, 1)).mean())


def fit_cnn(x, y, groups, train, valid, params, seed, epochs=None, use_aug=True):
    tf.keras.backend.clear_session()
    set_seed(seed)
    scaler = StandardScaler().fit(x[train].reshape(len(train), -1))
    def transform(indices):
        return scaler.transform(x[indices].reshape(len(indices), -1)).reshape((-1, POINTS, 9)).astype(np.float32)
    xx, yy = transform(train), y[train]
    if use_aug:
        xx, yy = augment(xx, yy, seed + 1)
    model = build_efnet(params)
    callbacks = [tf.keras.callbacks.TerminateOnNaN()]
    if valid is not None:
        callbacks += [
            PatientValidation(transform(valid), y[valid], groups[valid]),
            tf.keras.callbacks.EarlyStopping(monitor="val_patient_loss", mode="min",
                                            patience=PATIENCE, restore_best_weights=True),
        ]
    history = model.fit(xx, yy, epochs=epochs or EPOCHS, batch_size=BATCH_SIZE,
                        callbacks=callbacks, verbose=0)
    if not np.isfinite(history.history["loss"]).all():
        raise RuntimeError("CNN训练出现非有限损失；本次运行终止。")
    if valid is None:
        return model, scaler, int(epochs), history.history
    selected_epoch = int(np.argmin(history.history["val_patient_loss"]) + 1)
    return model, scaler, selected_epoch, history.history


def predict_model(name, model, scaler, x):
    if name == "EFNet":
        xx = scaler.transform(x.reshape(len(x), -1)).reshape((-1, POINTS, 9)).astype(np.float32)
        return np.asarray(model.predict(xx, batch_size=BATCH_SIZE, verbose=0))
    p = model.predict_proba(scalar_readout(x))
    return np.asarray(p[:, [list(model.classes_).index(i) for i in range(3)]])


def evaluate_candidate(name, params, x, y, groups, splits, seed):
    scores, epochs = [], []
    for j, (tr, va) in enumerate(splits):
        model_seed = seed + j
        if name == "EFNet":
            model, scaler, epoch, _ = fit_cnn(x, y, groups, tr, va, params, model_seed)
            epochs.append(epoch)
        else:
            model = make_classical(name, params, y[tr], groups[tr], model_seed)
            model.fit(scalar_readout(x[tr]), y[tr])
            scaler = None
        p = predict_model(name, model, scaler, x[va])
        _, truth, means = aggregate(p, y[va], groups[va])
        scores.append(macro_f1(truth, means))
        del model
    return {"score": float(np.mean(scores)), "inner_fold_macro_f1": scores,
            "refit_epochs": max(1, int(np.median(epochs))) if epochs else None,
            "inner_best_epochs": epochs}


def encode_candidates(candidates):
    columns = []
    for key in sorted(candidates[0]):
        vals = [p[key] for p in candidates]
        if all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in vals):
            column = np.asarray(vals, float)
            if key in {"C", "learning_rate", "l2"}:
                column = np.log10(column)
            columns.append(column)
        else:
            for category in sorted(set(map(repr, vals))):
                columns.append(np.array([repr(v) == category for v in vals], float))
    return StandardScaler().fit_transform(np.column_stack(columns))


def search_candidates(candidates, strategy, evaluate, seed, budget):
    """三种搜索共用同一候选集合；grid穷举，其余限额，费用差异写入输出。"""
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(candidates)).tolist()
    count = len(candidates) if strategy == "grid" else min(budget, len(candidates))
    encoded = encode_candidates(candidates)
    chosen, values, history = [], [], []
    for trial in range(count):
        acquisition = strategy
        if strategy == "grid":
            index = trial
        elif strategy == "random" or trial < 2:
            index = next(i for i in order if i not in chosen)
        else:
            unseen = [i for i in range(len(candidates)) if i not in chosen]
            # 固定核参数避免在极少候选观测上再过度拟合代理核。
            gp = GaussianProcessRegressor(kernel=Matern(nu=2.5) + WhiteKernel(1e-4),
                                          optimizer=None, normalize_y=True, alpha=1e-6,
                                          random_state=seed)
            gp.fit(encoded[chosen], np.asarray(values))
            mean, std = gp.predict(encoded[unseen], return_std=True)
            improvement = mean - max(values) - 0.01
            z = improvement / np.maximum(std, 1e-12)
            ei = improvement * norm.cdf(z) + std * norm.pdf(z)
            index = unseen[int(np.argmax(ei))]
            acquisition = "GP_expected_improvement"
        result = evaluate(candidates[index])
        if not np.isfinite(result["score"]):
            raise RuntimeError("内层搜索分数无效。")
        chosen.append(index)
        values.append(result["score"])
        history.append({"trial": trial + 1, "candidate_index": index, "params": candidates[index],
                        "acquisition": acquisition, **result})
    # 平分时按预先固定的候选编号决定；不看外层分数。
    best = min(history, key=lambda r: (-r["score"], r["candidate_index"]))
    return best, history


def add_predictions(patient_rows, experiment_rows, method, fold, prob, y, groups, mapping, test):
    ids, truth, means = aggregate(prob, y[test], groups[test])
    for patient, actual, p in zip(ids, truth, means):
        row = {"method": method, "fold": fold, "patient_id": patient,
               "true_code": int(actual), "pred_code": int(p.argmax()),
               "true_class": CLASSES[actual], "predicted_class": CLASSES[p.argmax()]}
        row.update({f"prob_{c}": float(p[j]) for j, c in enumerate(CLASSES)})
        patient_rows.append(row)
    for idx, p in zip(test, prob):
        row = {"method": method, "fold": fold, **mapping.iloc[idx].to_dict()}
        row.update({"predicted_class": CLASSES[p.argmax()],
                    **{f"prob_{c}": float(p[j]) for j, c in enumerate(CLASSES)}})
        experiment_rows.append(row)


def run_nested(x, y, groups, mapping, out):
    outer = patient_splits(np.arange(len(y)), y, groups, OUTER_K, SEED)
    patient_rows, experiment_rows, selection_rows, fold_rows, memberships = [], [], [], [], []
    budget = 2 if MODE == "smoke" else SEARCH_BUDGET
    folds_to_run = outer[:1] if MODE == "smoke" else outer
    for fold, (train, test) in enumerate(folds_to_run, 1):
        print(f"\n外层 {fold}/{len(folds_to_run)}，训练患者{len(set(groups[train]))}，测试患者{len(set(groups[test]))}", flush=True)
        inner = patient_splits(train, y, groups, INNER_K, SEED + fold)
        for j, (a, b) in enumerate([(train, test)] + inner):
            for role, indices in [("train", a), ("test" if j == 0 else "validation", b)]:
                for p in sorted(set(groups[indices])):
                    memberships.append({"outer_fold": fold, "inner_fold": j,
                                        "role": role, "patient_id": p, "class": p[0]})
        fold_rows.append({"fold": fold, "outer_train_patients": len(set(groups[train])),
                          "test_patients": len(set(groups[test])), "train_experiments": len(train),
                          "augmented_cnn_train_experiments": len(train) * (1 + AUG_FACTOR),
                          "test_experiments": len(test), "patient_overlap": 0,
                          **{f"test_{c}_patients": len(set(groups[test][y[test] == i]))
                             for i, c in enumerate(CLASSES)}})
        pd.DataFrame(memberships).to_csv(out / "patient_fold_assignments.csv", index=False)
        pd.DataFrame(fold_rows).to_csv(out / "fold_patient_counts.csv", index=False)
        # 多数类依据训练患者数，平局按W/Y/Z顺序，不从测试集确定。
        train_table = pd.DataFrame({"patient": groups[train], "y": y[train]}).drop_duplicates("patient")
        majority = np.bincount(train_table.y, minlength=3).argmax()
        prob = np.tile(np.eye(3)[majority], (len(test), 1))
        add_predictions(patient_rows, experiment_rows, "Majority", fold, prob, y, groups, mapping, test)
        choices = []
        cache = {}
        for model_no, name in enumerate(MODELS):
            candidates = list(ParameterGrid(GRIDS[name]))
            if MODE == "smoke":
                candidates = candidates[:2]
            def evaluate(params):
                key = (name, json.dumps(params, sort_keys=True))
                if key not in cache:
                    print(f"  {name} 内层评估 {params}", flush=True)
                    start = time.monotonic()
                    value = evaluate_candidate(name, params, x, y, groups, inner, SEED + fold * 100)
                    value["evaluation_seconds"] = time.monotonic() - start
                    cache[key] = value
                return cache[key]
            for strategy in SEARCHES:
                best, history = search_candidates(candidates, strategy, evaluate,
                                                  SEED + fold * 1000 + model_no, budget)
                choices.append({"model": name, "strategy": strategy, **best})
                folder = out / f"fold_{fold}" / f"{name}_{strategy}"
                folder.mkdir(parents=True, exist_ok=True)
                json_write(folder / "search_history.json", history)
                selection_rows.append({"fold": fold, "model": name, "strategy": strategy,
                                       "inner_macro_f1": best["score"], "evaluated_candidates": len(history),
                                       "total_candidate_space": len(candidates), "selected_epoch": best["refit_epochs"],
                                       "params": json.dumps(best["params"], sort_keys=True)})
        # 在接触任何外层模型分数之前，按内层结果选出一个完整选择流程的预测。
        winner = min(choices, key=lambda r: (-r["score"], MODELS.index(r["model"]),
                                             SEARCHES.index(r["strategy"])))
        json_write(out / f"fold_{fold}" / "inner_selected_method.json", winner)
        pd.DataFrame(selection_rows).to_csv(out / "selected_parameters.csv", index=False)
        refits, predictions = {}, {}
        for choice in choices:
            name, strategy, params = choice["model"], choice["strategy"], choice["params"]
            key = (name, json.dumps(params, sort_keys=True), choice["refit_epochs"])
            method = f"{name}_{strategy}"
            folder = out / f"fold_{fold}" / method
            if key not in refits:
                print(f"  外层重训 {name} {params}", flush=True)
                if name == "EFNet":
                    model, scaler, _, training_history = fit_cnn(
                        x, y, groups, train, None, params, SEED + fold * 100 + 50,
                        epochs=choice["refit_epochs"])
                    model.save(folder / "model.keras")
                    joblib.dump(scaler, folder / "scaler.joblib")
                    json_write(folder / "training_history.json", training_history)
                    json_write(folder / "capacity.json", {"parameters": model.count_params()})
                else:
                    model = make_classical(name, params, y[train], groups[train], SEED + fold * 100 + 50)
                    model.fit(scalar_readout(x[train]), y[train])
                    scaler = None
                    joblib.dump(model, folder / "model.joblib")
                prob = predict_model(name, model, scaler, x[test])
                refits[key] = (prob, folder)
                del model
            prob, source = refits[key]
            json_write(folder / "artifact_reference.json", {"directory": os.path.relpath(source, folder),
                       "model": name, "params": params, "class_order": CLASSES,
                       "scalar_readout": SCALAR_READOUT, "epochs": choice["refit_epochs"],
                       "scope": "outer-fold model, not a model fitted on all 22 patients"})
            predictions[method] = prob
            add_predictions(patient_rows, experiment_rows, method, fold, prob, y, groups, mapping, test)
        selected_prob = predictions[f"{winner['model']}_{winner['strategy']}"]
        add_predictions(patient_rows, experiment_rows, "InnerSelected", fold, selected_prob, y, groups, mapping, test)
        if RUN_NO_AUG and "EFNet" in MODELS:
            cnn_choices = [c for c in choices if c["model"] == "EFNet"]
            cnn_winner = min(cnn_choices, key=lambda r: (-r["score"], SEARCHES.index(r["strategy"])))
            add_predictions(patient_rows, experiment_rows, "EFNet_inner_selected_aug", fold,
                            predictions[f"EFNet_{cnn_winner['strategy']}"], y, groups, mapping, test)
            print("  EFNet配对无增强消融（相同参数、轮数、初始化种子）", flush=True)
            model, scaler, _, history = fit_cnn(
                x, y, groups, train, None, cnn_winner["params"], SEED + fold * 100 + 50,
                epochs=cnn_winner["refit_epochs"], use_aug=False)
            prob = predict_model("EFNet", model, scaler, x[test])
            folder = out / f"fold_{fold}" / "EFNet_no_aug"
            folder.mkdir()
            model.save(folder / "model.keras")
            joblib.dump(scaler, folder / "scaler.joblib")
            json_write(folder / "training_history.json", history)
            json_write(folder / "config.json", cnn_winner)
            add_predictions(patient_rows, experiment_rows, "EFNet_no_aug", fold, prob, y, groups, mapping, test)
            del model
        pd.DataFrame(patient_rows).to_csv(out / "patient_predictions.csv", index=False)
        pd.DataFrame(experiment_rows).to_csv(out / "experiment_predictions.csv", index=False)
    patients = pd.DataFrame(patient_rows)
    expected = set(groups[np.concatenate([test for _, test in folds_to_run])])
    for method, table in patients.groupby("method"):
        if table.patient_id.duplicated().any() or set(table.patient_id) != expected:
            raise RuntimeError(f"{method}患者预测覆盖错误。")
    return patients


def cp_ci(k, n):
    return (0.0 if k == 0 else float(beta.ppf(.025, k, n-k+1)),
            1.0 if k == n else float(beta.ppf(.975, k+1, n-k)))


def score_from_codes(y, pred):
    cm = confusion_matrix(y, pred, labels=[0, 1, 2]).astype(float)
    diagonal = cm.diagonal()
    precision = np.divide(diagonal, cm.sum(0), out=np.zeros(3), where=cm.sum(0) != 0)
    recall = np.divide(diagonal, cm.sum(1), out=np.zeros(3), where=cm.sum(1) != 0)
    f1 = np.divide(2*diagonal, cm.sum(0)+cm.sum(1), out=np.zeros(3), where=(cm.sum(0)+cm.sum(1)) != 0)
    return float((y == pred).mean()), float(f1.mean()), float(precision.mean()), float(recall.mean())


def bootstrap_indices(y):
    rng = np.random.default_rng(SEED + 9000)
    by_class = [np.flatnonzero(y == i) for i in range(3)]
    return [np.concatenate([rng.choice(idx, len(idx), replace=True) for idx in by_class])
            for _ in range(BOOTSTRAPS)]


def calibration_bins(y, prob):
    confidence, correct = prob.max(1), (prob.argmax(1) == y).astype(float)
    bins = np.minimum((confidence*4).astype(int), 3)
    rows = []
    for b in range(4):
        mask = bins == b
        rows.append({"lower": b/4, "upper": (b+1)/4, "count": int(mask.sum()),
                     "mean_confidence": float(confidence[mask].mean()) if mask.any() else np.nan,
                     "observed_accuracy": float(correct[mask].mean()) if mask.any() else np.nan})
    table = pd.DataFrame(rows)
    nonempty = table["count"] > 0
    ece = ((table.loc[nonempty, "count"] / len(y)) *
           abs(table.loc[nonempty, "mean_confidence"]-table.loc[nonempty, "observed_accuracy"])).sum()
    return table, float(ece)


def plot_diagnostics(method, y, prob, bins, out):
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    cm = confusion_matrix(y, prob.argmax(1), labels=[0, 1, 2])
    axes[0, 0].imshow(cm, cmap="Blues")
    for i, j in itertools.product(range(3), repeat=2):
        axes[0, 0].text(j, i, str(cm[i, j]), ha="center", va="center")
    axes[0, 0].set(xticks=range(3), yticks=range(3), xticklabels=CLASSES,
                   yticklabels=CLASSES, xlabel="Predicted", ylabel="True", title="Patient counts")
    valid = bins["count"] > 0
    axes[0, 1].plot([0, 1], [0, 1], "--", color="gray")
    axes[0, 1].plot(bins.loc[valid, "mean_confidence"], bins.loc[valid, "observed_accuracy"], "o-")
    axes[0, 1].set(xlim=(0, 1), ylim=(0, 1), xlabel="Top-label confidence",
                   ylabel="Observed accuracy", title="Descriptive calibration (4 fixed bins)")
    for i, c in enumerate(CLASSES):
        binary = (y == i).astype(int)
        fpr, tpr, _ = roc_curve(binary, prob[:, i])
        precision, recall, _ = precision_recall_curve(binary, prob[:, i])
        axes[1, 0].plot(fpr, tpr, label=c)
        axes[1, 1].plot(recall, precision, label=c)
    axes[1, 0].set(xlabel="False positive rate", ylabel="True positive rate", title="Patient ROC")
    axes[1, 1].set(xlabel="Recall", ylabel="Precision", title="Patient precision-recall")
    axes[1, 0].legend(); axes[1, 1].legend()
    fig.suptitle(f"{method}; n={len(y)}; {MODE}; fixed-panel internal evaluation")
    fig.tight_layout()
    fig.savefig(out / f"{method}.png", dpi=180)
    plt.close(fig)


def summarize(predictions, out):
    tables, metrics, recalls = {}, [], []
    for method, df in predictions.groupby("method", sort=True):
        df = df.sort_values("patient_id")
        y = df.true_code.to_numpy(int)
        prob = df[[f"prob_{c}" for c in CLASSES]].to_numpy(float)
        pred = prob.argmax(1)
        acc, f1, precision, recall = score_from_codes(y, pred)
        alo, ahi = cp_ci(int((y == pred).sum()), len(y))
        samples = bootstrap_indices(y)
        values = [score_from_codes(y[idx], pred[idx])[1] for idx in samples]
        flo, fhi = np.quantile(values, [.025, .975])
        bins, ece = calibration_bins(y, prob)
        bins.to_csv(out / f"calibration_bins_{method}.csv", index=False)
        # 每类概率校准表；与上面的top-label ECE定义不同，不能混用。
        class_bins = []
        for c, class_name in enumerate(CLASSES):
            assignments = np.minimum((prob[:, c]*4).astype(int), 3)
            for b in range(4):
                mask = assignments == b
                class_bins.append({"class": class_name, "bin": b, "count": int(mask.sum()),
                    "mean_probability": float(prob[mask, c].mean()) if mask.any() else np.nan,
                    "observed_fraction": float((y[mask] == c).mean()) if mask.any() else np.nan})
        pd.DataFrame(class_bins).to_csv(out / f"class_calibration_bins_{method}.csv", index=False)
        brier = float(np.mean(np.sum((prob-np.eye(3)[y])**2, axis=1)))
        loss = float(-np.log(np.clip(prob[np.arange(len(y)), y], 1e-12, 1)).mean())
        auc = roc_auc_score(y, prob, multi_class="ovr", average="macro", labels=[0, 1, 2])
        metrics.append({"method": method, "n_patients": len(y), "accuracy": acc,
                        "accuracy_ci_low": alo, "accuracy_ci_high": ahi, "macro_f1": f1,
                        "macro_f1_ci_low": float(flo), "macro_f1_ci_high": float(fhi),
                        "macro_precision": precision, "macro_recall_balanced_accuracy": recall,
                        "top_label_ece_4_bins": ece, "multiclass_brier_sum": brier,
                        "log_loss": loss, "macro_auc_ovr": auc})
        cm = confusion_matrix(y, pred, labels=[0, 1, 2])
        pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(out / f"confusion_{method}.csv")
        for i, c in enumerate(CLASSES):
            n = int((y == i).sum()); k = int(((y == i) & (pred == i)).sum())
            lo, hi = cp_ci(k, n)
            recalls.append({"method": method, "class": c, "correct": k, "n_patients": n,
                            "recall": k/n, "ci_low": lo, "ci_high": hi})
        tables[method] = (df.patient_id.to_numpy(), y, pred)
        if MAKE_PLOTS:
            plot_diagnostics(method, y, prob, bins, out)
    numeric = pd.DataFrame(metrics).sort_values("method")
    numeric.to_csv(out / "patient_metrics_numeric.csv", index=False)
    display = numeric.copy()
    for c in display.columns:
        if c not in {"method", "n_patients", "log_loss", "multiclass_brier_sum"}:
            display[c] = display[c].map(lambda v: f"{100*v:.1f}%")
    display.to_csv(out / "patient_metrics_paper.csv", index=False)
    pd.DataFrame(recalls).to_csv(out / "per_class_recall_ci.csv", index=False)
    # 预定参照；不根据外层最高分挑选参照。Holm修正本表的全部比较。
    pairs = []
    reference = "EFNet_inner_selected_aug" if "EFNet_inner_selected_aug" in tables else "InnerSelected"
    for method in tables:
        if method == reference:
            continue
        ids, y, a = tables[reference]; other_ids, other_y, b = tables[method]
        assert np.array_equal(ids, other_ids) and np.array_equal(y, other_y)
        won = int(((a == y) & (b != y)).sum())
        lost = int(((a != y) & (b == y)).sum())
        p = float(binomtest(won, won+lost, .5).pvalue) if won+lost else 1.0
        delta = []
        for idx in bootstrap_indices(y):
            aa = score_from_codes(y[idx], a[idx])
            bb = score_from_codes(y[idx], b[idx])
            delta.append([aa[0]-bb[0], aa[1]-bb[1]])
        intervals = np.quantile(delta, [.025, .975], axis=0)
        sa, sb = score_from_codes(y, a), score_from_codes(y, b)
        pairs.append({"reference": reference, "comparison": method, "ref_only_correct": won,
                      "comparison_only_correct": lost, "mcnemar_exact_p": p,
                      "accuracy_delta": sa[0]-sb[0], "accuracy_delta_low": intervals[0, 0],
                      "accuracy_delta_high": intervals[1, 0], "macro_f1_delta": sa[1]-sb[1],
                      "macro_f1_delta_low": intervals[0, 1], "macro_f1_delta_high": intervals[1, 1]})
    paired = pd.DataFrame(pairs)
    order = np.argsort(paired.mcnemar_exact_p.to_numpy())
    adjusted = np.minimum(1, np.maximum.accumulate(
        paired.mcnemar_exact_p.to_numpy()[order]*(len(paired)-np.arange(len(paired)))))
    paired["mcnemar_p_holm"] = 1.0
    paired.loc[order, "mcnemar_p_holm"] = adjusted
    paired.to_csv(out / "paired_comparisons_exploratory.csv", index=False)


def batch_audit(mapping, predictions, out):
    template = mapping[["patient_id", "true_class"]].drop_duplicates().sort_values("patient_id")
    for col in ["batch_id", "collection_date", "operator", "electrode_lot", "label_method"]:
        template[col] = ""
    template.to_csv(out / "patient_metadata_template.csv", index=False)
    if PATIENT_METADATA is None:
        json_write(out / "batch_audit.json", {"status": "not assessed: no metadata supplied"})
        return
    meta = pd.read_csv(PATIENT_METADATA, dtype=str).fillna("")
    expected = set(template.patient_id)
    if "patient_id" not in meta or meta.patient_id.duplicated().any() or set(meta.patient_id) != expected:
        raise ValueError("元数据必须每名患者一行，且与22名患者完全对应。")
    if "batch_id" not in meta or (meta.batch_id.str.strip() == "").any():
        json_write(out / "batch_audit.json", {"status": "not assessed: incomplete batch_id"})
        return
    meta["class"] = meta.patient_id.str[0]
    tab = pd.crosstab(meta.batch_id, meta["class"]).reindex(columns=CLASSES, fill_value=0)
    tab.to_csv(out / "batch_vs_class.csv")
    if len(tab) < 2:
        json_write(out / "batch_audit.json", {"status": "one batch only; batch effects not identifiable"})
        return
    batches = pd.Categorical(meta.batch_id).codes
    yy = meta["class"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy(int)
    def statistic(labels):
        counts = np.zeros((len(tab), 3))
        np.add.at(counts, (batches, labels), 1)
        exp = counts.sum(1, keepdims=True)*counts.sum(0, keepdims=True)/counts.sum()
        return float(((counts-exp)**2/exp).sum())
    observed = statistic(yy)
    rng = np.random.default_rng(SEED)
    exceed = sum(statistic(rng.permutation(yy)) >= observed-1e-12 for _ in range(BOOTSTRAPS))
    json_write(out / "batch_audit.json", {
        "status": "assessed", "chi_square_statistic": observed,
        "permutation_p": (exceed+1)/(BOOTSTRAPS+1),
        "all_batches_class_pure": bool((tab.gt(0).sum(1) == 1).all()),
        "note": "Small-sample descriptive check; non-significance cannot establish absence of batch effects."})


def input_tsne(x, y, groups, out):
    flat = x.reshape(len(x), -1)
    scaled = StandardScaler().fit_transform(flat)
    pca = PCA(n_components=min(30, len(x)-1), random_state=SEED).fit_transform(scaled)
    embedding = TSNE(n_components=2, perplexity=10, init="pca", learning_rate="auto",
                     random_state=SEED).fit_transform(pca)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(*embedding.T, c=y, cmap="viridis")
    codes = pd.Categorical(groups).codes
    axes[1].scatter(*embedding.T, c=codes, cmap="tab20")
    axes[0].set_title("Same input embedding: class colors")
    axes[1].set_title("Same input embedding: patient colors")
    fig.suptitle("Descriptive input-space t-SNE; never used for model selection")
    fig.tight_layout(); fig.savefig(out / "input_tsne.png", dpi=180); plt.close(fig)
    pd.DataFrame({"patient_id": groups, "class": [CLASSES[i] for i in y],
                  "tsne_1": embedding[:, 0], "tsne_2": embedding[:, 1]}).to_csv(
                      out / "experiment_input_tsne_coordinates.csv", index=False)
    ids = sorted(set(groups))
    patient_flat = np.stack([flat[groups == p].mean(0) for p in ids])
    patient_scaled = StandardScaler().fit_transform(patient_flat)
    patient_pca = PCA(n_components=min(15, len(ids)-1), random_state=SEED).fit_transform(patient_scaled)
    patient_embedding = TSNE(n_components=2, perplexity=5, init="pca", learning_rate="auto",
                             random_state=SEED).fit_transform(patient_pca)
    fig, ax = plt.subplots(figsize=(7, 6))
    for c, name in enumerate(CLASSES):
        mask = np.array([p.startswith(name) for p in ids])
        ax.scatter(*patient_embedding[mask].T, label=name)
    for p, point in zip(ids, patient_embedding):
        ax.annotate(p, point, fontsize=7)
    ax.legend(); ax.set_title("Patient-mean input t-SNE; descriptive only")
    fig.tight_layout(); fig.savefig(out / "patient_input_tsne.png", dpi=180); plt.close(fig)
    pd.DataFrame({"patient_id": ids, "tsne_1": patient_embedding[:, 0],
                  "tsne_2": patient_embedding[:, 1]}).to_csv(out / "patient_input_tsne_coordinates.csv", index=False)


def main():
    global EPOCHS, BOOTSTRAPS
    if MODE not in {"smoke", "full"}:
        raise ValueError("MODE应为smoke或full")
    if not set(MODELS) <= set(GRIDS) or not MODELS or set(SEARCHES) != {"grid", "random", "bayesian"}:
        raise ValueError("检查MODELS和SEARCHES设置。")
    if MODE == "smoke":
        EPOCHS, BOOTSTRAPS = 1, 100
    set_seed(SEED)
    tf.config.experimental.enable_op_determinism()
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(gpu, True)
    # 每次新建目录，不覆盖已完成的结果；不自动把既往外层结果当作新实验。
    out = OUTPUT_ROOT / (MODE + "_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(parents=True, exist_ok=False)
    status = {"mode": MODE, "status": "running", "valid_for_manuscript": False}
    json_write(out / "run_status.json", status)
    try:
        x, y, groups, mapping, voltage = load_data(DATA_FILE)
        mapping.to_csv(out / "experiment_patient_mapping.csv", index=False)
        scalar = pd.concat([mapping, pd.DataFrame(scalar_readout(x), columns=SENSORS)], axis=1)
        scalar.to_csv(out / "scalar_features.csv", index=False)
        pd.DataFrame({"point": np.arange(1, POINTS+1), "voltage": voltage}).to_csv(out / "common_voltage.csv", index=False)
        versions = {p: importlib.metadata.version(p) for p in
                    ["numpy", "pandas", "scipy", "scikit-learn", "tensorflow", "matplotlib", "joblib"]}
        plan = {
            "mode": MODE, "seed": SEED, "outer_k": OUTER_K, "inner_k": INNER_K,
            "patient_counts": {"W": 5, "Y": 8, "Z": 9}, "experiment_count": len(x),
            "sensor_rows": len(x)*9, "models": MODELS, "searches": SEARCHES, "grids": GRIDS,
            "random_bayesian_budget": 2 if MODE == "smoke" else SEARCH_BUDGET,
            "grid_budget": "exhaustive over declared grid (not equal-cost comparison)",
            "candidate_cache": "Identical candidate evaluations shared across search strategies; GP only sees queried scores",
            "primary_metric": "patient macro-F1", "search_objective": "mean inner-fold patient macro-F1",
            "aggregation": "mean of 3 replicate probability vectors, then argmax",
            "epoch_selection": "median of inner best patient-loss epochs, refit from scratch",
            "max_epochs": EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE,
            "augmentation": {"extra_copies": AUG_FACTOR, "noise_std": NOISE_STD, "scale_amp": SCALE_AMP},
            "no_aug_ablation": "same augmented-model inner-selected parameters/epochs; no retuning",
            "scalar_readout": SCALAR_READOUT, "input_shape": [POINTS, 9], "sensor_order": SENSORS,
            "class_order": CLASSES, "bootstrap_replicates": BOOTSTRAPS,
            "ece": "top-label, four equal-width bins on [0,1]",
            "ci_scope": "Clopper-Pearson binomial summaries; class-stratified bootstrap of fixed OOF patient predictions",
            "search_seeds": "SEED + outer_fold*1000 + model_index",
            "training_seeds": "inner: SEED+outer_fold*100+inner_index; refit: SEED+outer_fold*100+50",
            "data_sha256": file_hash(DATA_FILE),
            "metadata_sha256": file_hash(PATIENT_METADATA) if PATIENT_METADATA is not None else None,
            "python": sys.version, "versions": versions, "limitation": LIMITATION,
        }
        json_write(out / "analysis_plan.json", plan)
        json_write(out / "input_tensor_definition.json", {"shape": [POINTS, 9], "channels": SENSORS,
                   "class_order": CLASSES, "voltage": "shared coordinate, excluded as predictor",
                   "scalar_readout": SCALAR_READOUT})
        predictions = run_nested(x, y, groups, mapping, out)
        summarize(predictions, out)
        batch_audit(mapping, predictions, out)
        if MAKE_TSNE and MODE == "full":
            input_tsne(x, y, groups, out)
        status.update(status="complete", valid_for_manuscript=MODE == "full",
                      scope="internal fixed-panel evaluation only", limitation=LIMITATION)
        json_write(out / "run_status.json", status)
        print(f"\n完成：{out.resolve()}", flush=True)
        if MODE == "smoke":
            print("SMOKE结果只用于检查程序，不能作为论文结果；正式运行需MODE='full'。")
    except Exception as exc:
        status.update(status="failed", error=repr(exc))
        json_write(out / "run_status.json", status)
        raise


if __name__ == "__main__":
    main()



外层 1/5，训练患者17，测试患者5
  LDA 内层评估 {'shrinkage': None, 'solver': 'lsqr'}
  LDA 内层评估 {'shrinkage': 'auto', 'solver': 'lsqr'}
  LDA 内层评估 {'shrinkage': 0.1, 'solver': 'lsqr'}
  LDA 内层评估 {'shrinkage': 0.5, 'solver': 'lsqr'}
  LDA 内层评估 {'shrinkage': 0.9, 'solver': 'lsqr'}
  SVM 内层评估 {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
  SVM 内层评估 {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
  SVM 内层评估 {'C': 1.0, 'gamma': 'scale', 'kernel': 'linear'}
  SVM 内层评估 {'C': 1.0, 'gamma': 'scale', 'kernel': 'rbf'}
  SVM 内层评估 {'C': 10.0, 'gamma': 'scale', 'kernel': 'linear'}
  SVM 内层评估 {'C': 10.0, 'gamma': 'scale', 'kernel': 'rbf'}
  RF 内层评估 {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 100}
  RF 内层评估 {'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 100}
  RF 内层评估 {'max_depth': 3, 'min_samples_leaf': 1, 'n_estimators': 100}
  RF 内层评估 {'max_depth': 3, 'min_samples_leaf': 2, 'n_estimators': 100}
  RF 内层评估 {'max_depth': 5, 'min_samples_leaf': 1, 'n_estimators': 100}
  RF 内层评估 {'max_depth

In [10]:
from pathlib import Path

result_dir = Path(
    "/content/drive/MyDrive/snakebite_patient_cv_results/"
    "full_20261007_023718_227996"
)

if not result_dir.exists():
    print("结果目录不存在，请检查Drive是否已挂载，以及目录名称是否正确。")
else:
    print("该结果目录中的JSON文件：")
    for path in sorted(result_dir.glob("*.json")):
        print(path.name)

    target = result_dir / "analysis_plan.json"

    if target.exists():
        print("\n已找到：", target)
        print(target.read_text(encoding="utf-8"))
    else:
        print("\n当前目录未找到，搜索其他运行目录：")
        root = result_dir.parent
        matches = sorted(root.rglob("analysis_plan.json"))

        for path in matches:
            print(path)

        if not matches:
            print("所有运行目录中均未找到 analysis_plan.json。")

该结果目录中的JSON文件：
analysis_plan.json
batch_audit.json
input_tensor_definition.json
run_status.json

已找到： /content/drive/MyDrive/snakebite_patient_cv_results/full_20261007_023718_227996/analysis_plan.json
{
  "mode": "full",
  "seed": 42,
  "outer_k": 5,
  "inner_k": 3,
  "patient_counts": {
    "W": 5,
    "Y": 8,
    "Z": 9
  },
  "experiment_count": 66,
  "sensor_rows": 594,
  "models": [
    "LDA",
    "SVM",
    "RF",
    "DT",
    "KNN",
    "GB",
    "Logistic",
    "EFNet"
  ],
  "searches": [
    "grid",
    "random",
    "bayesian"
  ],
  "grids": {
    "LDA": {
      "solver": [
        "lsqr"
      ],
      "shrinkage": [
        null,
        "auto",
        0.1,
        0.5,
        0.9
      ]
    },
    "SVM": {
      "C": [
        0.1,
        1.0,
        10.0
      ],
      "kernel": [
        "linear",
        "rbf"
      ],
      "gamma": [
        "scale"
      ]
    },
    "RF": {
      "n_estimators": [
        100
      ],
      "max_depth": [
        null,
      

In [11]:
from google.colab import drive, files

drive.mount("/content/drive")
files.upload()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving patient_level_nested_cv_colab.py to patient_level_nested_cv_colab.py
Saving electrochem_data1.csv to electrochem_data1.csv


{'patient_level_nested_cv_colab.py': b'"""\n\xe5\x9b\xba\xe5\xae\x9a9\xe6\xa0\x87\xe5\xbf\x97\xe7\x89\xa9\xe9\x9d\xa2\xe6\x9d\xbf\xef\xbc\x9a\xe6\x82\xa3\xe8\x80\x85\xe5\xb9\xb3\xe5\x9d\x87\xe6\xb3\xa2\xe5\xbd\xa2\xe7\x9a\x84\xe6\x82\xa3\xe8\x80\x85\xe7\xba\xa7\xe5\xb5\x8c\xe5\xa5\x97\xe4\xba\xa4\xe5\x8f\x89\xe9\xaa\x8c\xe8\xaf\x81\xef\xbc\x88Python 3.10+\xef\xbc\x89\n\xe8\xbe\x93\xe5\x85\xa5\xef\xbc\x9a\xe5\xb7\xb2\xe7\xbb\x8f\xe5\xae\x8c\xe6\x88\x90\xe4\xb8\x89\xe6\xac\xa1\xe6\xb5\x8b\xe9\x87\x8f\xe5\xb9\xb3\xe5\x9d\x87\xe7\x9a\x84198\xe8\xa1\x8cCSV\xef\xbc\x8822\xe5\x90\x8d\xe6\x82\xa3\xe8\x80\x85\xc3\x979\xe6\x9d\xa1\xe4\xbc\xa0\xe6\x84\x9f\xe5\x99\xa8\xe6\xb3\xa2\xe5\xbd\xa2\xef\xbc\x89\xe3\x80\x82\n\xe4\xbb\xa3\xe7\xa0\x81\xe4\xb8\x8d\xe5\x86\x8d\xe8\xae\xa1\xe7\xae\x97\xe5\x9d\x87\xe5\x80\xbc\xef\xbc\x8c\xe7\x9b\xb4\xe6\x8e\xa5\xe5\xb0\x86\xe6\xaf\x8f\xe5\x90\x8d\xe6\x82\xa3\xe8\x80\x85\xe7\x9a\x849\xe6\x9d\xa1\xe5\xb9\xb3\xe5\x9d\x87\xe6\xb3\xa2\xe5\xbd\xa2\xe7\xbb\x84\xe6\x88\

In [1]:
"""
固定9标志物面板：患者平均波形的患者级嵌套交叉验证（Python 3.10+）
输入：已经完成三次测量平均的198行CSV（22名患者×9条传感器波形）。
代码不再计算均值，直接将每名患者的9条平均波形组成一个患者级输入；原始594行数据不需要上传。
输出：每种模型/搜索策略的患者预测、指标/区间、配对比较、模型和审计记录。
重要：面板来自同一发现队列，本脚本不能消除上游面板选择偏差。
默认 MODE="smoke" 仅检查程序；正式分析改为 MODE="full"，完整运行后才可报告结果。
Bayesian 为有限候选集合上的 Gaussian process + expected improvement 搜索。
"""
from __future__ import annotations
import hashlib
import importlib.metadata
import itertools
import json
import os
import random
import re
import sys
import time
from datetime import datetime
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.stats import beta, binomtest, norm
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import (
    confusion_matrix, f1_score,
    precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.model_selection import ParameterGrid, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# ========================= 只需先检查这里 =========================
MODE = "full"
DATA_FILE = Path("/content/electrochem_data1.csv")
OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/snakebite_patient_cv_results_average"
)
PATIENT_METADATA = None             # 可选 Path("patient_metadata.csv")
SEED = 42
OUTER_K, INNER_K = 5, 3
MODELS = ["LDA", "SVM", "RF", "DT", "KNN", "GB", "Logistic", "EFNet"]
SEARCHES = ["grid", "random", "bayesian"]
SEARCH_BUDGET = 4                   # random/bayesian最多候选数；grid穷举，预算可能较大
EPOCHS, PATIENCE, BATCH_SIZE = 200, 25, 8
BOOTSTRAPS = 2000
RUN_NO_AUG = True                   # 沿用相应增强CNN所选参数和轮数，配对去除增强
MAKE_PLOTS = True
MAKE_TSNE = True
AUG_FACTOR, NOISE_STD, SCALE_AMP = 2, 0.008, 0.08
SCALAR_READOUT = "peak_abs"          # 单个预定读数：每通道整个扫描的最大绝对电流
# 若实验已规定固定电位/峰区间读数，应在正式运行前修改scalar_readout，
# 并更新说明；不要看外层测试结果后挑选读数。
SENSORS = ["FGG", "FGB", "CP", "FETUB", "HBD", "CPN2", "AGT", "LUM", "ACTB"]
CLASSES = ["W", "Y", "Z"]
POINTS = 2200
# 只调少量参数；三种搜索是选择策略，并非三种新的分类器。
GRIDS = {
    "LDA": {"solver": ["lsqr"], "shrinkage": [None, "auto", 0.1, 0.5, 0.9]},
    "SVM": {"C": [0.1, 1.0, 10.0], "kernel": ["linear", "rbf"], "gamma": ["scale"]},
    "RF": {"n_estimators": [100], "max_depth": [None, 3, 5], "min_samples_leaf": [1, 2]},
    "DT": {"max_depth": [2, 3, 5], "min_samples_leaf": [1, 2]},
    "KNN": {"n_neighbors": [3, 5, 7], "weights": ["uniform", "distance"]},
    "GB": {"n_estimators": [50, 100], "learning_rate": [0.05, 0.1], "max_depth": [1, 2]},
    "Logistic": {"C": [0.01, 0.1, 1.0, 10.0, 100.0]},
    "EFNet": {"learning_rate": [3e-5, 1e-4, 3e-4], "l2": [1e-4, 1e-3]},
}
LIMITATION = (
    "Fixed-panel internal exploratory evaluation. The same 22 patients contributed "
    "to panel discovery; upstream selection bias is not removed. CIs and paired tests "
    "describe cross-fitted patient predictions and do not capture all training/selection "
    "uncertainty. Outer-score rankings are exploratory, not independent confirmation "
    "of a post-hoc selected winner."
)


def json_write(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False,
                                   default=lambda x: x.item() if isinstance(x, np.generic) else str(x)),
                          encoding="utf-8")


def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


def load_data(path):
    """Load an already averaged patient-level CSV; do not average inside this script."""
    df = pd.read_csv(path)
    df = df.drop(columns=[c for c in df if str(c).startswith("Unnamed")])
    cc = [f"current_dot{i}" for i in range(1, POINTS + 1)]
    vc = [f"voltage_dot{i}" for i in range(1, POINTS + 1)]
    missing = set(["type", "sensor_type"] + cc + vc) - set(df)
    if missing:
        raise ValueError(f"缺少列：{sorted(missing)[:10]}")
    if len(df) != 198:
        raise ValueError(
            f"本脚本要求已经完成三次测量平均的198行CSV（22患者×9传感器），实际为{len(df)}行。"
        )
    df["type"] = df["type"].astype(str).str.strip().str.upper()
    df["sensor_type"] = df["sensor_type"].astype(str).str.strip()
    parsed = df["type"].str.extract(r"^([WYZ]\d+)(?:_(?:MEAN3|MEAN|AVG|AVERAGE|0|1))?$", expand=True)
    if parsed.isna().any().any():
        raise ValueError("平均数据的type应为W1、W1_mean3、W1_avg或保留的W1_1等患者编号格式。")
    df["patient_id"] = parsed[0]
    if set(df.sensor_type) != set(SENSORS):
        raise ValueError("sensor_type必须是指定9个蛋白。")
    current = df[cc].apply(pd.to_numeric, errors="coerce")
    current = current.interpolate(axis=1, limit_direction="both").ffill(axis=1).bfill(axis=1)
    voltage = df[vc].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    if not np.isfinite(current.to_numpy()).all() or not np.isfinite(voltage).all():
        raise ValueError("波形存在无法补齐的缺失值或无穷值。")
    if not np.array_equal(voltage, np.broadcast_to(voltage[0], voltage.shape)):
        raise ValueError("电压轨迹并非完全相同，需要先明确对齐方法。")
    tensors, rows = [], []
    for patient_id, sub in df.groupby("patient_id", sort=True):
        if len(sub) != 9 or sub.sensor_type.nunique() != 9:
            raise ValueError(f"{patient_id}必须包含9个传感器且各一次。")
        idx = sub.assign(_row=sub.index).set_index("sensor_type").loc[SENSORS, "_row"].to_numpy()
        tensors.append(current.loc[idx].to_numpy(np.float32).T)
        rows.append({"patient_id": patient_id, "experiment_id": f"{patient_id}_mean3",
                     "replicate_id": 0, "n_replicates_averaged": 3,
                     "true_class": patient_id[0]})
    mapping = pd.DataFrame(rows)
    counts = mapping.true_class.value_counts().to_dict()
    if counts != {"W": 5, "Y": 8, "Z": 9}:
        raise ValueError(f"患者数应为W/Y/Z=5/8/9，实际为{counts}")
    x = np.stack(tensors)
    y = mapping.true_class.map({c: i for i, c in enumerate(CLASSES)}).to_numpy(int)
    groups = mapping.patient_id.to_numpy()
    return x, y, groups, mapping, voltage[0]


def scalar_readout(x):
    if SCALAR_READOUT != "peak_abs":
        raise ValueError("请实现并说明选定的标量读数规则。")
    return np.max(np.abs(x), axis=1).astype(float)


def patient_splits(indices, y, groups, k, seed):
    """先在一人一行的表上分层，再映射回实验行，避免分组近似分层失败。"""
    table = pd.DataFrame({"patient": groups[indices], "y": y[indices]})
    if table.groupby("patient").y.nunique().max() != 1:
        raise ValueError("同一患者存在多个标签。")
    table = table.drop_duplicates("patient").sort_values("patient").reset_index(drop=True)
    if table.y.value_counts().min() < k:
        raise ValueError(f"每类独立患者不足以进行{k}折分层。")
    cv = StratifiedKFold(k, shuffle=True, random_state=seed)
    splits = []
    for tr, va in cv.split(table.patient, table.y):
        a = indices[np.isin(groups[indices], table.patient.iloc[tr])]
        b = indices[np.isin(groups[indices], table.patient.iloc[va])]
        assert set(groups[a]).isdisjoint(groups[b])
        assert set(y[a]) == set(y[b]) == {0, 1, 2}
        splits.append((a, b))
    return splits


def aggregate(prob, y, groups):
    if not np.isfinite(prob).all() or not np.allclose(prob.sum(1), 1, atol=1e-5):
        raise ValueError("预测概率无效。")
    ids = sorted(set(groups))
    truth = np.array([y[groups == p][0] for p in ids])
    means = np.stack([prob[groups == p].mean(0) for p in ids])
    return ids, truth, means


def macro_f1(truth, prob):
    return float(f1_score(truth, prob.argmax(1), labels=[0, 1, 2],
                          average="macro", zero_division=0))


def make_classical(name, params, train_y, train_groups, seed):
    factories = {
        "LDA": lambda: LinearDiscriminantAnalysis(**params),
        "RF": lambda: RandomForestClassifier(**params, max_features="sqrt", random_state=seed, n_jobs=1),
        "DT": lambda: DecisionTreeClassifier(**params, random_state=seed),
        "KNN": lambda: KNeighborsClassifier(**params),
        "GB": lambda: GradientBoostingClassifier(**params, random_state=seed),
        "Logistic": lambda: LogisticRegression(**params, max_iter=5000, solver="lbfgs", random_state=seed),
    }
    if name == "SVM":
        # 禁用SVC隐含的记录级概率校准；显式以患者分组校准，scaler也在校准折内拟合。
        splits = patient_splits(np.arange(len(train_y)), train_y, train_groups, 2, seed)
        base = make_pipeline(StandardScaler(), SVC(**params, probability=False, random_state=seed))
        return CalibratedClassifierCV(estimator=base, method="sigmoid", cv=splits, ensemble=False)
    return make_pipeline(StandardScaler(), factories[name]())


def build_efnet(params):
    reg = tf.keras.regularizers.l2(params["l2"])
    kernels = [15, 13, 11, 9, 7, 5, 3, 1]
    def rf(ks):
        field, jump = 1, 1
        for k in ks:
            field += k * jump
            jump *= 2
        return field
    while rf(kernels) < 2000:
        kernels.append(1)
    model = tf.keras.Sequential([tf.keras.layers.Input((POINTS, len(SENSORS)))])
    for i, k in enumerate(kernels):
        filters = [64, 96, 256][i] if i < 3 else 320
        model.add(tf.keras.layers.Conv1D(filters, k, padding="same", activation="elu", kernel_regularizer=reg))
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.MaxPooling1D(2))
    model.add(tf.keras.layers.GlobalAveragePooling1D())
    model.add(tf.keras.layers.Dense(128, activation="elu", kernel_regularizer=reg))
    model.add(tf.keras.layers.Dropout(0.6))
    model.add(tf.keras.layers.Dense(3, activation="softmax", kernel_regularizer=reg))
    model.compile(optimizer=tf.keras.optimizers.Adam(params["learning_rate"]),
                  loss="sparse_categorical_crossentropy")
    return model


def augment(x, y, seed):
    rng = np.random.default_rng(seed)
    xs, ys = [x], [y]
    for _ in range(AUG_FACTOR):
        noise = rng.normal(0, NOISE_STD, x.shape).astype(np.float32)
        scale = rng.uniform(1-SCALE_AMP, 1+SCALE_AMP, (len(x), 1, 1)).astype(np.float32)
        xs.append((x + noise) * scale)
        ys.append(y)
    xx, yy = np.concatenate(xs), np.concatenate(ys)
    order = rng.permutation(len(xx))
    return xx[order], yy[order]


class PatientValidation(tf.keras.callbacks.Callback):
    """以患者平均概率的交叉熵监控早停。只接收内层验证患者。"""
    def __init__(self, x, y, groups):
        super().__init__()
        self.x, self.y, self.groups = x, y, groups

    def on_epoch_end(self, epoch, logs=None):
        p = np.asarray(self.model(self.x, training=False))
        _, truth, prob = aggregate(p, self.y, self.groups)
        logs["val_patient_loss"] = float(-np.log(np.clip(prob[np.arange(len(truth)), truth], 1e-12, 1)).mean())


def fit_cnn(x, y, groups, train, valid, params, seed, epochs=None, use_aug=True):
    tf.keras.backend.clear_session()
    set_seed(seed)
    scaler = StandardScaler().fit(x[train].reshape(len(train), -1))
    def transform(indices):
        return scaler.transform(x[indices].reshape(len(indices), -1)).reshape((-1, POINTS, 9)).astype(np.float32)
    xx, yy = transform(train), y[train]
    if use_aug:
        xx, yy = augment(xx, yy, seed + 1)
    model = build_efnet(params)
    callbacks = [tf.keras.callbacks.TerminateOnNaN()]
    if valid is not None:
        callbacks += [
            PatientValidation(transform(valid), y[valid], groups[valid]),
            tf.keras.callbacks.EarlyStopping(monitor="val_patient_loss", mode="min",
                                            patience=PATIENCE, restore_best_weights=True),
        ]
    history = model.fit(xx, yy, epochs=epochs or EPOCHS, batch_size=BATCH_SIZE,
                        callbacks=callbacks, verbose=0)
    if not np.isfinite(history.history["loss"]).all():
        raise RuntimeError("CNN训练出现非有限损失；本次运行终止。")
    if valid is None:
        return model, scaler, int(epochs), history.history
    selected_epoch = int(np.argmin(history.history["val_patient_loss"]) + 1)
    return model, scaler, selected_epoch, history.history


def predict_model(name, model, scaler, x):
    if name == "EFNet":
        xx = scaler.transform(x.reshape(len(x), -1)).reshape((-1, POINTS, 9)).astype(np.float32)
        return np.asarray(model.predict(xx, batch_size=BATCH_SIZE, verbose=0))
    p = model.predict_proba(scalar_readout(x))
    return np.asarray(p[:, [list(model.classes_).index(i) for i in range(3)]])


def evaluate_candidate(name, params, x, y, groups, splits, seed):
    scores, epochs = [], []
    for j, (tr, va) in enumerate(splits):
        model_seed = seed + j
        if name == "EFNet":
            model, scaler, epoch, _ = fit_cnn(x, y, groups, tr, va, params, model_seed)
            epochs.append(epoch)
        else:
            model = make_classical(name, params, y[tr], groups[tr], model_seed)
            model.fit(scalar_readout(x[tr]), y[tr])
            scaler = None
        p = predict_model(name, model, scaler, x[va])
        _, truth, means = aggregate(p, y[va], groups[va])
        scores.append(macro_f1(truth, means))
        del model
    return {"score": float(np.mean(scores)), "inner_fold_macro_f1": scores,
            "refit_epochs": max(1, int(np.median(epochs))) if epochs else None,
            "inner_best_epochs": epochs}


def encode_candidates(candidates):
    columns = []
    for key in sorted(candidates[0]):
        vals = [p[key] for p in candidates]
        if all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in vals):
            column = np.asarray(vals, float)
            if key in {"C", "learning_rate", "l2"}:
                column = np.log10(column)
            columns.append(column)
        else:
            for category in sorted(set(map(repr, vals))):
                columns.append(np.array([repr(v) == category for v in vals], float))
    return StandardScaler().fit_transform(np.column_stack(columns))


def search_candidates(candidates, strategy, evaluate, seed, budget):
    """三种搜索共用同一候选集合；grid穷举，其余限额，费用差异写入输出。"""
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(candidates)).tolist()
    count = len(candidates) if strategy == "grid" else min(budget, len(candidates))
    encoded = encode_candidates(candidates)
    chosen, values, history = [], [], []
    for trial in range(count):
        acquisition = strategy
        if strategy == "grid":
            index = trial
        elif strategy == "random" or trial < 2:
            index = next(i for i in order if i not in chosen)
        else:
            unseen = [i for i in range(len(candidates)) if i not in chosen]
            # 固定核参数避免在极少候选观测上再过度拟合代理核。
            gp = GaussianProcessRegressor(kernel=Matern(nu=2.5) + WhiteKernel(1e-4),
                                          optimizer=None, normalize_y=True, alpha=1e-6,
                                          random_state=seed)
            gp.fit(encoded[chosen], np.asarray(values))
            mean, std = gp.predict(encoded[unseen], return_std=True)
            improvement = mean - max(values) - 0.01
            z = improvement / np.maximum(std, 1e-12)
            ei = improvement * norm.cdf(z) + std * norm.pdf(z)
            index = unseen[int(np.argmax(ei))]
            acquisition = "GP_expected_improvement"
        result = evaluate(candidates[index])
        if not np.isfinite(result["score"]):
            raise RuntimeError("内层搜索分数无效。")
        chosen.append(index)
        values.append(result["score"])
        history.append({"trial": trial + 1, "candidate_index": index, "params": candidates[index],
                        "acquisition": acquisition, **result})
    # 平分时按预先固定的候选编号决定；不看外层分数。
    best = min(history, key=lambda r: (-r["score"], r["candidate_index"]))
    return best, history


def add_predictions(patient_rows, experiment_rows, method, fold, prob, y, groups, mapping, test):
    ids, truth, means = aggregate(prob, y[test], groups[test])
    for patient, actual, p in zip(ids, truth, means):
        row = {"method": method, "fold": fold, "patient_id": patient,
               "true_code": int(actual), "pred_code": int(p.argmax()),
               "true_class": CLASSES[actual], "predicted_class": CLASSES[p.argmax()]}
        row.update({f"prob_{c}": float(p[j]) for j, c in enumerate(CLASSES)})
        patient_rows.append(row)
    for idx, p in zip(test, prob):
        row = {"method": method, "fold": fold, **mapping.iloc[idx].to_dict()}
        row.update({"predicted_class": CLASSES[p.argmax()],
                    **{f"prob_{c}": float(p[j]) for j, c in enumerate(CLASSES)}})
        experiment_rows.append(row)


def run_nested(x, y, groups, mapping, out):
    outer = patient_splits(np.arange(len(y)), y, groups, OUTER_K, SEED)
    patient_rows, experiment_rows, selection_rows, fold_rows, memberships = [], [], [], [], []
    budget = 2 if MODE == "smoke" else SEARCH_BUDGET
    folds_to_run = outer[:1] if MODE == "smoke" else outer
    for fold, (train, test) in enumerate(folds_to_run, 1):
        print(f"\n外层 {fold}/{len(folds_to_run)}，训练患者{len(set(groups[train]))}，测试患者{len(set(groups[test]))}", flush=True)
        inner = patient_splits(train, y, groups, INNER_K, SEED + fold)
        for j, (a, b) in enumerate([(train, test)] + inner):
            for role, indices in [("train", a), ("test" if j == 0 else "validation", b)]:
                for p in sorted(set(groups[indices])):
                    memberships.append({"outer_fold": fold, "inner_fold": j,
                                        "role": role, "patient_id": p, "class": p[0]})
        fold_rows.append({"fold": fold, "outer_train_patients": len(set(groups[train])),
                          "test_patients": len(set(groups[test])), "train_patient_average_inputs": len(train),
                          "augmented_cnn_train_patient_inputs": len(train) * (1 + AUG_FACTOR),
                          "test_patient_average_inputs": len(test), "patient_overlap": 0,
                          **{f"test_{c}_patients": len(set(groups[test][y[test] == i]))
                             for i, c in enumerate(CLASSES)}})
        pd.DataFrame(memberships).to_csv(out / "patient_fold_assignments.csv", index=False)
        pd.DataFrame(fold_rows).to_csv(out / "fold_patient_counts.csv", index=False)
        # 多数类依据训练患者数，平局按W/Y/Z顺序，不从测试集确定。
        train_table = pd.DataFrame({"patient": groups[train], "y": y[train]}).drop_duplicates("patient")
        majority = np.bincount(train_table.y, minlength=3).argmax()
        prob = np.tile(np.eye(3)[majority], (len(test), 1))
        add_predictions(patient_rows, experiment_rows, "Majority", fold, prob, y, groups, mapping, test)
        choices = []
        cache = {}
        for model_no, name in enumerate(MODELS):
            candidates = list(ParameterGrid(GRIDS[name]))
            if MODE == "smoke":
                candidates = candidates[:2]
            def evaluate(params):
                key = (name, json.dumps(params, sort_keys=True))
                if key not in cache:
                    print(f"  {name} 内层评估 {params}", flush=True)
                    start = time.monotonic()
                    value = evaluate_candidate(name, params, x, y, groups, inner, SEED + fold * 100)
                    value["evaluation_seconds"] = time.monotonic() - start
                    cache[key] = value
                return cache[key]
            for strategy in SEARCHES:
                best, history = search_candidates(candidates, strategy, evaluate,
                                                  SEED + fold * 1000 + model_no, budget)
                choices.append({"model": name, "strategy": strategy, **best})
                folder = out / f"fold_{fold}" / f"{name}_{strategy}"
                folder.mkdir(parents=True, exist_ok=True)
                json_write(folder / "search_history.json", history)
                selection_rows.append({"fold": fold, "model": name, "strategy": strategy,
                                       "inner_macro_f1": best["score"], "evaluated_candidates": len(history),
                                       "total_candidate_space": len(candidates), "selected_epoch": best["refit_epochs"],
                                       "params": json.dumps(best["params"], sort_keys=True)})
        # 在接触任何外层模型分数之前，按内层结果选出一个完整选择流程的预测。
        winner = min(choices, key=lambda r: (-r["score"], MODELS.index(r["model"]),
                                             SEARCHES.index(r["strategy"])))
        json_write(out / f"fold_{fold}" / "inner_selected_method.json", winner)
        pd.DataFrame(selection_rows).to_csv(out / "selected_parameters.csv", index=False)
        refits, predictions = {}, {}
        for choice in choices:
            name, strategy, params = choice["model"], choice["strategy"], choice["params"]
            key = (name, json.dumps(params, sort_keys=True), choice["refit_epochs"])
            method = f"{name}_{strategy}"
            folder = out / f"fold_{fold}" / method
            if key not in refits:
                print(f"  外层重训 {name} {params}", flush=True)
                if name == "EFNet":
                    model, scaler, _, training_history = fit_cnn(
                        x, y, groups, train, None, params, SEED + fold * 100 + 50,
                        epochs=choice["refit_epochs"])
                    model.save(folder / "model.keras")
                    joblib.dump(scaler, folder / "scaler.joblib")
                    json_write(folder / "training_history.json", training_history)
                    json_write(folder / "capacity.json", {"parameters": model.count_params()})
                else:
                    model = make_classical(name, params, y[train], groups[train], SEED + fold * 100 + 50)
                    model.fit(scalar_readout(x[train]), y[train])
                    scaler = None
                    joblib.dump(model, folder / "model.joblib")
                prob = predict_model(name, model, scaler, x[test])
                refits[key] = (prob, folder)
                del model
            prob, source = refits[key]
            json_write(folder / "artifact_reference.json", {"directory": os.path.relpath(source, folder),
                       "model": name, "params": params, "class_order": CLASSES,
                       "scalar_readout": SCALAR_READOUT, "epochs": choice["refit_epochs"],
                       "scope": "outer-fold model, not a model fitted on all 22 patients"})
            predictions[method] = prob
            add_predictions(patient_rows, experiment_rows, method, fold, prob, y, groups, mapping, test)
        selected_prob = predictions[f"{winner['model']}_{winner['strategy']}"]
        add_predictions(patient_rows, experiment_rows, "InnerSelected", fold, selected_prob, y, groups, mapping, test)
        if RUN_NO_AUG and "EFNet" in MODELS:
            cnn_choices = [c for c in choices if c["model"] == "EFNet"]
            cnn_winner = min(cnn_choices, key=lambda r: (-r["score"], SEARCHES.index(r["strategy"])))
            add_predictions(patient_rows, experiment_rows, "EFNet_inner_selected_aug", fold,
                            predictions[f"EFNet_{cnn_winner['strategy']}"], y, groups, mapping, test)
            print("  EFNet配对无增强消融（相同参数、轮数、初始化种子）", flush=True)
            model, scaler, _, history = fit_cnn(
                x, y, groups, train, None, cnn_winner["params"], SEED + fold * 100 + 50,
                epochs=cnn_winner["refit_epochs"], use_aug=False)
            prob = predict_model("EFNet", model, scaler, x[test])
            folder = out / f"fold_{fold}" / "EFNet_no_aug"
            folder.mkdir()
            model.save(folder / "model.keras")
            joblib.dump(scaler, folder / "scaler.joblib")
            json_write(folder / "training_history.json", history)
            json_write(folder / "config.json", cnn_winner)
            add_predictions(patient_rows, experiment_rows, "EFNet_no_aug", fold, prob, y, groups, mapping, test)
            del model
        pd.DataFrame(patient_rows).to_csv(out / "patient_predictions.csv", index=False)
        pd.DataFrame(experiment_rows).to_csv(out / "patient_average_predictions.csv", index=False)
    patients = pd.DataFrame(patient_rows)
    expected = set(groups[np.concatenate([test for _, test in folds_to_run])])
    for method, table in patients.groupby("method"):
        if table.patient_id.duplicated().any() or set(table.patient_id) != expected:
            raise RuntimeError(f"{method}患者预测覆盖错误。")
    return patients


def cp_ci(k, n):
    return (0.0 if k == 0 else float(beta.ppf(.025, k, n-k+1)),
            1.0 if k == n else float(beta.ppf(.975, k+1, n-k)))


def score_from_codes(y, pred):
    cm = confusion_matrix(y, pred, labels=[0, 1, 2]).astype(float)
    diagonal = cm.diagonal()
    precision = np.divide(diagonal, cm.sum(0), out=np.zeros(3), where=cm.sum(0) != 0)
    recall = np.divide(diagonal, cm.sum(1), out=np.zeros(3), where=cm.sum(1) != 0)
    f1 = np.divide(2*diagonal, cm.sum(0)+cm.sum(1), out=np.zeros(3), where=(cm.sum(0)+cm.sum(1)) != 0)
    return float((y == pred).mean()), float(f1.mean()), float(precision.mean()), float(recall.mean())


def bootstrap_indices(y):
    rng = np.random.default_rng(SEED + 9000)
    by_class = [np.flatnonzero(y == i) for i in range(3)]
    return [np.concatenate([rng.choice(idx, len(idx), replace=True) for idx in by_class])
            for _ in range(BOOTSTRAPS)]


def calibration_bins(y, prob):
    confidence, correct = prob.max(1), (prob.argmax(1) == y).astype(float)
    bins = np.minimum((confidence*4).astype(int), 3)
    rows = []
    for b in range(4):
        mask = bins == b
        rows.append({"lower": b/4, "upper": (b+1)/4, "count": int(mask.sum()),
                     "mean_confidence": float(confidence[mask].mean()) if mask.any() else np.nan,
                     "observed_accuracy": float(correct[mask].mean()) if mask.any() else np.nan})
    table = pd.DataFrame(rows)
    nonempty = table["count"] > 0
    ece = ((table.loc[nonempty, "count"] / len(y)) *
           abs(table.loc[nonempty, "mean_confidence"]-table.loc[nonempty, "observed_accuracy"])).sum()
    return table, float(ece)


def plot_diagnostics(method, y, prob, bins, out):
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    cm = confusion_matrix(y, prob.argmax(1), labels=[0, 1, 2])
    axes[0, 0].imshow(cm, cmap="Blues")
    for i, j in itertools.product(range(3), repeat=2):
        axes[0, 0].text(j, i, str(cm[i, j]), ha="center", va="center")
    axes[0, 0].set(xticks=range(3), yticks=range(3), xticklabels=CLASSES,
                   yticklabels=CLASSES, xlabel="Predicted", ylabel="True", title="Patient counts")
    valid = bins["count"] > 0
    axes[0, 1].plot([0, 1], [0, 1], "--", color="gray")
    axes[0, 1].plot(bins.loc[valid, "mean_confidence"], bins.loc[valid, "observed_accuracy"], "o-")
    axes[0, 1].set(xlim=(0, 1), ylim=(0, 1), xlabel="Top-label confidence",
                   ylabel="Observed accuracy", title="Descriptive calibration (4 fixed bins)")
    for i, c in enumerate(CLASSES):
        binary = (y == i).astype(int)
        fpr, tpr, _ = roc_curve(binary, prob[:, i])
        precision, recall, _ = precision_recall_curve(binary, prob[:, i])
        axes[1, 0].plot(fpr, tpr, label=c)
        axes[1, 1].plot(recall, precision, label=c)
    axes[1, 0].set(xlabel="False positive rate", ylabel="True positive rate", title="Patient ROC")
    axes[1, 1].set(xlabel="Recall", ylabel="Precision", title="Patient precision-recall")
    axes[1, 0].legend(); axes[1, 1].legend()
    fig.suptitle(f"{method}; n={len(y)}; {MODE}; fixed-panel internal evaluation")
    fig.tight_layout()
    fig.savefig(out / f"{method}.png", dpi=180)
    plt.close(fig)


def summarize(predictions, out):
    tables, metrics, recalls = {}, [], []
    for method, df in predictions.groupby("method", sort=True):
        df = df.sort_values("patient_id")
        y = df.true_code.to_numpy(int)
        prob = df[[f"prob_{c}" for c in CLASSES]].to_numpy(float)
        pred = prob.argmax(1)
        acc, f1, precision, recall = score_from_codes(y, pred)
        alo, ahi = cp_ci(int((y == pred).sum()), len(y))
        samples = bootstrap_indices(y)
        values = [score_from_codes(y[idx], pred[idx])[1] for idx in samples]
        flo, fhi = np.quantile(values, [.025, .975])
        bins, ece = calibration_bins(y, prob)
        bins.to_csv(out / f"calibration_bins_{method}.csv", index=False)
        # 每类概率校准表；与上面的top-label ECE定义不同，不能混用。
        class_bins = []
        for c, class_name in enumerate(CLASSES):
            assignments = np.minimum((prob[:, c]*4).astype(int), 3)
            for b in range(4):
                mask = assignments == b
                class_bins.append({"class": class_name, "bin": b, "count": int(mask.sum()),
                    "mean_probability": float(prob[mask, c].mean()) if mask.any() else np.nan,
                    "observed_fraction": float((y[mask] == c).mean()) if mask.any() else np.nan})
        pd.DataFrame(class_bins).to_csv(out / f"class_calibration_bins_{method}.csv", index=False)
        brier = float(np.mean(np.sum((prob-np.eye(3)[y])**2, axis=1)))
        loss = float(-np.log(np.clip(prob[np.arange(len(y)), y], 1e-12, 1)).mean())
        auc = roc_auc_score(y, prob, multi_class="ovr", average="macro", labels=[0, 1, 2])
        metrics.append({"method": method, "n_patients": len(y), "accuracy": acc,
                        "accuracy_ci_low": alo, "accuracy_ci_high": ahi, "macro_f1": f1,
                        "macro_f1_ci_low": float(flo), "macro_f1_ci_high": float(fhi),
                        "macro_precision": precision, "macro_recall_balanced_accuracy": recall,
                        "top_label_ece_4_bins": ece, "multiclass_brier_sum": brier,
                        "log_loss": loss, "macro_auc_ovr": auc})
        cm = confusion_matrix(y, pred, labels=[0, 1, 2])
        pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(out / f"confusion_{method}.csv")
        for i, c in enumerate(CLASSES):
            n = int((y == i).sum()); k = int(((y == i) & (pred == i)).sum())
            lo, hi = cp_ci(k, n)
            recalls.append({"method": method, "class": c, "correct": k, "n_patients": n,
                            "recall": k/n, "ci_low": lo, "ci_high": hi})
        tables[method] = (df.patient_id.to_numpy(), y, pred)
        if MAKE_PLOTS:
            plot_diagnostics(method, y, prob, bins, out)
    numeric = pd.DataFrame(metrics).sort_values("method")
    numeric.to_csv(out / "patient_metrics_numeric.csv", index=False)
    display = numeric.copy()
    for c in display.columns:
        if c not in {"method", "n_patients", "log_loss", "multiclass_brier_sum"}:
            display[c] = display[c].map(lambda v: f"{100*v:.1f}%")
    display.to_csv(out / "patient_metrics_paper.csv", index=False)
    pd.DataFrame(recalls).to_csv(out / "per_class_recall_ci.csv", index=False)
    # 预定参照；不根据外层最高分挑选参照。Holm修正本表的全部比较。
    pairs = []
    reference = "EFNet_inner_selected_aug" if "EFNet_inner_selected_aug" in tables else "InnerSelected"
    for method in tables:
        if method == reference:
            continue
        ids, y, a = tables[reference]; other_ids, other_y, b = tables[method]
        assert np.array_equal(ids, other_ids) and np.array_equal(y, other_y)
        won = int(((a == y) & (b != y)).sum())
        lost = int(((a != y) & (b == y)).sum())
        p = float(binomtest(won, won+lost, .5).pvalue) if won+lost else 1.0
        delta = []
        for idx in bootstrap_indices(y):
            aa = score_from_codes(y[idx], a[idx])
            bb = score_from_codes(y[idx], b[idx])
            delta.append([aa[0]-bb[0], aa[1]-bb[1]])
        intervals = np.quantile(delta, [.025, .975], axis=0)
        sa, sb = score_from_codes(y, a), score_from_codes(y, b)
        pairs.append({"reference": reference, "comparison": method, "ref_only_correct": won,
                      "comparison_only_correct": lost, "mcnemar_exact_p": p,
                      "accuracy_delta": sa[0]-sb[0], "accuracy_delta_low": intervals[0, 0],
                      "accuracy_delta_high": intervals[1, 0], "macro_f1_delta": sa[1]-sb[1],
                      "macro_f1_delta_low": intervals[0, 1], "macro_f1_delta_high": intervals[1, 1]})
    paired = pd.DataFrame(pairs)
    order = np.argsort(paired.mcnemar_exact_p.to_numpy())
    adjusted = np.minimum(1, np.maximum.accumulate(
        paired.mcnemar_exact_p.to_numpy()[order]*(len(paired)-np.arange(len(paired)))))
    paired["mcnemar_p_holm"] = 1.0
    paired.loc[order, "mcnemar_p_holm"] = adjusted
    paired.to_csv(out / "paired_comparisons_exploratory.csv", index=False)


def batch_audit(mapping, predictions, out):
    template = mapping[["patient_id", "true_class"]].drop_duplicates().sort_values("patient_id")
    for col in ["batch_id", "collection_date", "operator", "electrode_lot", "label_method"]:
        template[col] = ""
    template.to_csv(out / "patient_metadata_template.csv", index=False)
    if PATIENT_METADATA is None:
        json_write(out / "batch_audit.json", {"status": "not assessed: no metadata supplied"})
        return
    meta = pd.read_csv(PATIENT_METADATA, dtype=str).fillna("")
    expected = set(template.patient_id)
    if "patient_id" not in meta or meta.patient_id.duplicated().any() or set(meta.patient_id) != expected:
        raise ValueError("元数据必须每名患者一行，且与22名患者完全对应。")
    if "batch_id" not in meta or (meta.batch_id.str.strip() == "").any():
        json_write(out / "batch_audit.json", {"status": "not assessed: incomplete batch_id"})
        return
    meta["class"] = meta.patient_id.str[0]
    tab = pd.crosstab(meta.batch_id, meta["class"]).reindex(columns=CLASSES, fill_value=0)
    tab.to_csv(out / "batch_vs_class.csv")
    if len(tab) < 2:
        json_write(out / "batch_audit.json", {"status": "one batch only; batch effects not identifiable"})
        return
    batches = pd.Categorical(meta.batch_id).codes
    yy = meta["class"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy(int)
    def statistic(labels):
        counts = np.zeros((len(tab), 3))
        np.add.at(counts, (batches, labels), 1)
        exp = counts.sum(1, keepdims=True)*counts.sum(0, keepdims=True)/counts.sum()
        return float(((counts-exp)**2/exp).sum())
    observed = statistic(yy)
    rng = np.random.default_rng(SEED)
    exceed = sum(statistic(rng.permutation(yy)) >= observed-1e-12 for _ in range(BOOTSTRAPS))
    json_write(out / "batch_audit.json", {
        "status": "assessed", "chi_square_statistic": observed,
        "permutation_p": (exceed+1)/(BOOTSTRAPS+1),
        "all_batches_class_pure": bool((tab.gt(0).sum(1) == 1).all()),
        "note": "Small-sample descriptive check; non-significance cannot establish absence of batch effects."})


def input_tsne(x, y, groups, out):
    flat = x.reshape(len(x), -1)
    scaled = StandardScaler().fit_transform(flat)
    pca = PCA(n_components=min(30, len(x)-1), random_state=SEED).fit_transform(scaled)
    embedding = TSNE(n_components=2, perplexity=10, init="pca", learning_rate="auto",
                     random_state=SEED).fit_transform(pca)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(*embedding.T, c=y, cmap="viridis")
    codes = pd.Categorical(groups).codes
    axes[1].scatter(*embedding.T, c=codes, cmap="tab20")
    axes[0].set_title("Same input embedding: class colors")
    axes[1].set_title("Same input embedding: patient colors")
    fig.suptitle("Descriptive patient-average input-space t-SNE; never used for model selection")
    fig.tight_layout(); fig.savefig(out / "input_tsne.png", dpi=180); plt.close(fig)
    pd.DataFrame({"patient_id": groups, "class": [CLASSES[i] for i in y],
                  "tsne_1": embedding[:, 0], "tsne_2": embedding[:, 1]}).to_csv(
                      out / "patient_average_input_tsne_coordinates.csv", index=False)
    ids = sorted(set(groups))
    patient_flat = np.stack([flat[groups == p].mean(0) for p in ids])
    patient_scaled = StandardScaler().fit_transform(patient_flat)
    patient_pca = PCA(n_components=min(15, len(ids)-1), random_state=SEED).fit_transform(patient_scaled)
    patient_embedding = TSNE(n_components=2, perplexity=5, init="pca", learning_rate="auto",
                             random_state=SEED).fit_transform(patient_pca)
    fig, ax = plt.subplots(figsize=(7, 6))
    for c, name in enumerate(CLASSES):
        mask = np.array([p.startswith(name) for p in ids])
        ax.scatter(*patient_embedding[mask].T, label=name)
    for p, point in zip(ids, patient_embedding):
        ax.annotate(p, point, fontsize=7)
    ax.legend(); ax.set_title("Patient-average input t-SNE; descriptive only")
    fig.tight_layout(); fig.savefig(out / "patient_input_tsne.png", dpi=180); plt.close(fig)
    pd.DataFrame({"patient_id": ids, "tsne_1": patient_embedding[:, 0],
                  "tsne_2": patient_embedding[:, 1]}).to_csv(out / "patient_input_tsne_coordinates.csv", index=False)


def main():
    global EPOCHS, BOOTSTRAPS
    if MODE not in {"smoke", "full"}:
        raise ValueError("MODE应为smoke或full")
    if not set(MODELS) <= set(GRIDS) or not MODELS or set(SEARCHES) != {"grid", "random", "bayesian"}:
        raise ValueError("检查MODELS和SEARCHES设置。")
    if MODE == "smoke":
        EPOCHS, BOOTSTRAPS = 1, 100
    set_seed(SEED)
    tf.config.experimental.enable_op_determinism()
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(gpu, True)
    # 每次新建目录，不覆盖已完成的结果；不自动把既往外层结果当作新实验。
    out = OUTPUT_ROOT / (MODE + "_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(parents=True, exist_ok=False)
    status = {"mode": MODE, "status": "running", "valid_for_manuscript": False}
    json_write(out / "run_status.json", status)
    try:
        x, y, groups, mapping, voltage = load_data(DATA_FILE)
        mapping.to_csv(out / "patient_average_input_mapping.csv", index=False)
        scalar = pd.concat([mapping, pd.DataFrame(scalar_readout(x), columns=SENSORS)], axis=1)
        scalar.to_csv(out / "scalar_features.csv", index=False)
        pd.DataFrame({"point": np.arange(1, POINTS+1), "voltage": voltage}).to_csv(out / "common_voltage.csv", index=False)
        versions = {p: importlib.metadata.version(p) for p in
                    ["numpy", "pandas", "scipy", "scikit-learn", "tensorflow", "matplotlib", "joblib"]}
        plan = {
            "mode": MODE, "seed": SEED, "outer_k": OUTER_K, "inner_k": INNER_K,
            "patient_counts": {"W": 5, "Y": 8, "Z": 9}, "patient_average_input_count": len(x),
            "averaged_sensor_rows": len(x)*9, "source_mode": "externally pre-averaged three-repeat waveforms",
            "source_expected_rows": 198, "models": MODELS, "searches": SEARCHES, "grids": GRIDS,
            "random_bayesian_budget": 2 if MODE == "smoke" else SEARCH_BUDGET,
            "grid_budget": "exhaustive over declared grid (not equal-cost comparison)",
            "candidate_cache": "Identical candidate evaluations shared across search strategies; GP only sees queried scores",
            "primary_metric": "patient macro-F1", "search_objective": "mean inner-fold patient macro-F1",
            "aggregation": "none: one pre-averaged waveform input per patient; one prediction per patient",
            "epoch_selection": "median of inner best patient-loss epochs, refit from scratch",
            "max_epochs": EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE,
            "augmentation": {"extra_copies": AUG_FACTOR, "noise_std": NOISE_STD, "scale_amp": SCALE_AMP},
            "no_aug_ablation": "same augmented-model inner-selected parameters/epochs; no retuning",
            "scalar_readout": SCALAR_READOUT, "input_shape": [POINTS, 9], "sensor_order": SENSORS,
            "class_order": CLASSES, "bootstrap_replicates": BOOTSTRAPS,
            "ece": "top-label, four equal-width bins on [0,1]",
            "ci_scope": "Clopper-Pearson binomial summaries; class-stratified bootstrap of fixed OOF patient predictions",
            "search_seeds": "SEED + outer_fold*1000 + model_index",
            "training_seeds": "inner: SEED+outer_fold*100+inner_index; refit: SEED+outer_fold*100+50",
            "data_sha256": file_hash(DATA_FILE),
            "metadata_sha256": file_hash(PATIENT_METADATA) if PATIENT_METADATA is not None else None,
            "python": sys.version, "versions": versions, "limitation": LIMITATION,
        }
        json_write(out / "analysis_plan.json", plan)
        json_write(out / "input_tensor_definition.json", {"shape": [POINTS, 9], "channels": SENSORS,
                   "class_order": CLASSES, "voltage": "shared coordinate, excluded as predictor",
                   "scalar_readout": SCALAR_READOUT,
                   "input_definition": "one externally pre-averaged waveform per patient",
                   "source_rows": 198})
        predictions = run_nested(x, y, groups, mapping, out)
        summarize(predictions, out)
        batch_audit(mapping, predictions, out)
        if MAKE_TSNE and MODE == "full":
            input_tsne(x, y, groups, out)
        status.update(status="complete", valid_for_manuscript=MODE == "full",
                      scope="internal fixed-panel evaluation only", limitation=LIMITATION)
        json_write(out / "run_status.json", status)
        print(f"\n完成：{out.resolve()}", flush=True)
        if MODE == "smoke":
            print("SMOKE结果只用于检查程序，不能作为论文结果；正式运行需MODE='full'。")
    except Exception as exc:
        status.update(status="failed", error=repr(exc))
        json_write(out / "run_status.json", status)
        raise


if __name__ == "__main__":
    main()



外层 1/5，训练患者17，测试患者5
  LDA 内层评估 {'shrinkage': None, 'solver': 'lsqr'}
  LDA 内层评估 {'shrinkage': 'auto', 'solver': 'lsqr'}
  LDA 内层评估 {'shrinkage': 0.1, 'solver': 'lsqr'}
  LDA 内层评估 {'shrinkage': 0.5, 'solver': 'lsqr'}
  LDA 内层评估 {'shrinkage': 0.9, 'solver': 'lsqr'}
  SVM 内层评估 {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
  SVM 内层评估 {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
  SVM 内层评估 {'C': 1.0, 'gamma': 'scale', 'kernel': 'linear'}
  SVM 内层评估 {'C': 1.0, 'gamma': 'scale', 'kernel': 'rbf'}
  SVM 内层评估 {'C': 10.0, 'gamma': 'scale', 'kernel': 'linear'}
  SVM 内层评估 {'C': 10.0, 'gamma': 'scale', 'kernel': 'rbf'}
  RF 内层评估 {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 100}
  RF 内层评估 {'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 100}
  RF 内层评估 {'max_depth': 3, 'min_samples_leaf': 1, 'n_estimators': 100}
  RF 内层评估 {'max_depth': 3, 'min_samples_leaf': 2, 'n_estimators': 100}
  RF 内层评估 {'max_depth': 5, 'min_samples_leaf': 1, 'n_estimators': 100}
  RF 内层评估 {'max_depth

  EFNet 内层评估 {'l2': 0.0001, 'learning_rate': 0.0003}
  EFNet 内层评估 {'l2': 0.001, 'learning_rate': 3e-05}
  EFNet 内层评估 {'l2': 0.001, 'learning_rate': 0.0001}
  EFNet 内层评估 {'l2': 0.001, 'learning_rate': 0.0003}
  外层重训 LDA {'shrinkage': 'auto', 'solver': 'lsqr'}
  外层重训 SVM {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
  外层重训 SVM {'C': 1.0, 'gamma': 'scale', 'kernel': 'rbf'}
  外层重训 SVM {'C': 1.0, 'gamma': 'scale', 'kernel': 'linear'}
  外层重训 RF {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 100}
  外层重训 DT {'max_depth': 2, 'min_samples_leaf': 1}
  外层重训 DT {'max_depth': 2, 'min_samples_leaf': 2}
  外层重训 KNN {'n_neighbors': 3, 'weights': 'uniform'}
  外层重训 KNN {'n_neighbors': 5, 'weights': 'uniform'}
  外层重训 KNN {'n_neighbors': 3, 'weights': 'distance'}
  外层重训 GB {'learning_rate': 0.05, 'max_depth': 1, 'n_estimators': 50}
  外层重训 Logistic {'C': 0.1}
  外层重训 Logistic {'C': 1.0}
  外层重训 EFNet {'l2': 0.0001, 'learning_rate': 0.0001}
  EFNet配对无增强消融（相同参数、轮数、初始化种子）

外层 2/5，训练患者17，测试患者5
  LDA